In [1]:
import pandas as pd

import data_retrieval.order_data as order_data
from data_retrieval.order_data import get_prices_from_station
import data_retrieval.historical_data as historical_data
import data_processing.render as render
from data_processing.sde_utils import get_item_names, get_item_sizes
import os

from data_retrieval.price_differences import get_ungrouped_prices_from_station, get_volume_price_history
import settings

import nest_asyncio
from datetime import datetime, timezone

CLIENT_ID and CLIENT_SECRET found, proceeding.
loaded settings:
Settings(cache_all=False, hide_existing=False, source_hub='Jita', source_order_type='sell', dest_hub='1DQ1-A', dest_order_type='buy', days_history='7', bogus_threshold='10')


In [2]:
# Fix for Jupyter notebook
nest_asyncio.apply()
pd.set_option('display.width', 1000)

minimum_margin = 0.1 # 10%

### basically a duplicate of price_differences.py. There is probably a better way to do this.  
# give penalty score for competing sellers if settings.dest_order_type == sell
# Add sorting to output html. Maybe output as excel or csv? 
# Maybe add those variables below to settings? 

#def price_differences(settings):
print('Loading Static Data Export mappings...', end=' ')
typeIDs_to_names = get_item_names()
#typeIDs_to_sizes = get_item_sizes()
print('Done!')


Loading Static Data Export mappings... Done!


In [3]:

# Results DataFrame
summary = pd.DataFrame()


In [4]:
cache_orders = True
# Order Series
buy_prices = get_prices_from_station("Jita", "buy", cache_orders)
sell_prices = get_prices_from_station("Jita", "sell", cache_orders)
ungrouped_buy_prices = get_ungrouped_prices_from_station("Jita", "buy", cache_orders)
ungrouped_sell_prices = get_ungrouped_prices_from_station("Jita", "sell", cache_orders)

# TODO need to add perimeter citadels in a way that makes sense

#print(str(buy_prices))
#print(str(ungrouped_buy_prices))


Loading buy orders from Jita using cached data. Attempt 1...
Getting Jita buy orders from redis cache
Got 88323 orders!
Loading sell orders from Jita using cached data. Attempt 1...
Getting Jita sell orders from redis cache
Got 207037 orders!
Loading buy orders from Jita using cached data...
Getting Jita buy orders from redis cache
Got 88323 orders!
       duration  is_buy_order                issued  location_id  min_volume    order_id        price    range  system_id  type_id  volume_remain  volume_total
0            90          True  2024-11-03T11:30:52Z     60003760           1  6905921586     11000.00  station   30000142     6717            827          1000
1            90          True  2024-09-16T00:56:20Z     60003760           1  6855589942      4612.00  station   30000142    63643             10            12
2            90          True  2024-11-04T12:33:20Z     60003760           1  6905921604    737100.00        1   30000142    33710             68            68
3       

In [5]:
print(ungrouped_buy_prices)

       duration  is_buy_order                issued  location_id  min_volume    order_id        price    range  system_id  type_id  volume_remain  volume_total
0            90          True  2024-11-03T11:30:52Z     60003760           1  6905921586     11000.00  station   30000142     6717            827          1000
1            90          True  2024-09-16T00:56:20Z     60003760           1  6855589942      4612.00  station   30000142    63643             10            12
2            90          True  2024-11-04T12:33:20Z     60003760           1  6905921604    737100.00        1   30000142    33710             68            68
3            90          True  2024-08-10T18:32:22Z     60003760           1  6815744080       510.30  station   30000142    22983          48342         50000
4            90          True  2024-11-04T15:30:35Z     60003760           1  6863803747 165000000.00  station   30000142    46666              1             1
...         ...           ...           

In [6]:
ungrouped_buy_prices.loc[ungrouped_buy_prices['type_id'] == 14692]
print(ungrouped_buy_prices.loc[ungrouped_buy_prices['type_id'] == 14692])

       duration  is_buy_order                issued  location_id  min_volume    order_id        price    range  system_id  type_id  volume_remain  volume_total
14396        90          True  2024-10-31T20:30:35Z     60003760           1  6866237077 408000000.00   region   30000142    14692              2             2
16922        90          True  2024-11-02T01:35:52Z     60003760           1  6876593674 409900000.00  station   30000142    14692              1             1
21447        90          True  2024-10-17T18:27:54Z     60003760           1  6777862295 267100000.00  station   30000142    14692              2             2
26703        90          True  2024-11-02T08:15:29Z     60003760           1  6889460441 410000000.00  station   30000142    14692              1             1
34247        90          True  2024-10-27T17:09:11Z     60003760           1  6902140989 407500000.00  station   30000142    14692              1             1
55306        90          True  2024-09-0

In [7]:
def analyze_order_timing(df,order_type):
    """
    Analyze market orders to show how many orders were issued within different time ranges
    and calculate an activity score.
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing market orders with 'issued' and 'type_id' columns
    
    Returns:
    pandas.DataFrame: Analysis of orders per time range and activity score, indexed by type_id
    """
    # Convert issued column to datetime if it's not already
    df['issued'] = pd.to_datetime(df['issued'])
    
    # Get current time in UTC
    current_time = datetime.now(timezone.utc)
    
    # Define time ranges and their scores
    time_ranges = [
        ('last_1h', pd.Timedelta(hours=1), 10),
        ('last_4h', pd.Timedelta(hours=4), 5),
        ('last_12h', pd.Timedelta(hours=12), 3),
        ('last_24h', pd.Timedelta(hours=24), 2),
        ('last_48h', pd.Timedelta(hours=48), 1)
    ]
    
    # Create a dictionary to store results
    results = {}
    
    # Calculate counts for each time range (non-overlapping)
    previous_cutoff = None
    
    for range_name, time_delta, score in time_ranges:
        cutoff_time = current_time - time_delta
        
        if previous_cutoff is None:
            # For the first range (0 to 1h)
            mask = df['issued'] >= cutoff_time
        else:
            # For subsequent ranges (e.g., 1h to 4h)
            mask = (df['issued'] >= cutoff_time) & (df['issued'] < previous_cutoff)
        
        counts = df[mask].groupby('type_id').size()
        results[range_name] = counts
        
        # Store this cutoff time for the next iteration
        previous_cutoff = cutoff_time
    
    # Combine all results into a single DataFrame
    result_df = pd.DataFrame(results)
    
    # Fill NaN values with 0 since they represent no orders in that time range
    result_df = result_df.fillna(0).astype(int)
    
    # Calculate the score
    scores = {
        'last_1h': 10,
        'last_4h': 5,
        'last_12h': 3,
        'last_24h': 2,
        'last_48h': 1
    }
    if order_type == 'buy':
        result_df['buy_competition'] = sum(result_df[col] * scores[col] for col in scores.keys())
    if order_type == 'sell':
        result_df['sell_competition'] = sum(result_df[col] * scores[col] for col in scores.keys())
    
    return result_df

In [8]:
sorted_buy_orders = analyze_order_timing(ungrouped_buy_prices,'buy')
sorted_sell_orders = analyze_order_timing(ungrouped_sell_prices,'sell')
sorted_orders = sorted_buy_orders.merge(sorted_sell_orders[['sell_competition']], how='left', left_index=True, right_index=True)


In [9]:
print(sorted_orders)
print(sorted_orders.loc[48418])

         last_1h  last_4h  last_12h  last_24h  last_48h  buy_competition  sell_competition
type_id                                                                                   
18             0        1         0         0         2                7              5.00
19             0        0         0         1         0                2               NaN
20             0        0         2         1         1                9             20.00
21             0        0         0         2         0                4               NaN
22             0        1         0         1         2                9             12.00
...          ...      ...       ...       ...       ...              ...               ...
86150          0        0         0         0         1                1               NaN
86151          0        1         0         0         0                5             10.00
86154          0        0         0         0         1                1              2.00

In [10]:
# Sell-side costs: Broker fees (~3.5%), Sales tax (~2.25%), Hauling fees (~2%).
sell_prices_taxed = sell_prices * ((1 - 0.035 - 0.0225) * (1 - 0.02))

# Series back into output DF columns
summary['Buy price'] = buy_prices
summary['Sell price'] = sell_prices_taxed
summary['Profit'] = summary['Sell price'] - summary['Buy price']
summary['Margin'] = summary['Profit'] / summary['Buy price']

# Group the remaining rows by type_id
grouped_buy_prices = ungrouped_buy_prices.groupby('type_id').agg({'price':'mean','volume_remain':'sum'})
print("grouped_buy_prices")
print(str(grouped_buy_prices))

summary = summary.merge(sorted_orders[['buy_competition','sell_competition']], how='left', on='type_id')

grouped_buy_prices.to_csv('exports/grouped_buy_prices.csv')  
# Save the total volume_remaining for each type_id
summary['Units available'] = grouped_buy_prices['volume_remain']
# Remove rows below margin treshold
summary = summary[summary['Margin'] >= minimum_margin]

#summary = summary.fillna(0).astype(int)

# Print summary DataFrame to check values
print("summary:")
print(str(summary))
# Get historical volume and price for order refinement


grouped_buy_prices
               price  volume_remain
type_id                            
18             38.61       27590697
19          13526.67          35857
20            750.35        4136472
21            882.52         161667
22           5432.00          70625
...              ...            ...
86152    39755000.00              8
86153   725410357.14             35
86154   184000000.00              7
86155     5468620.00             20
86156   245266666.67              6

[15749 rows x 2 columns]
summary:
            Buy price    Sell price        Profit  Margin  buy_competition  sell_competition  Units available
type_id                                                                                                      
18              42.98         55.42         12.44    0.29             7.00              5.00         27590697
19           15010.00      27700.26      12690.26    0.85             2.00               NaN            35857
21            1015.00       1616.39   

In [11]:
volume, price, order_count = get_volume_price_history("Jita", summary.index, 30, False)

summary['Daily Volume'] = volume
summary['Avg Price'] = price
summary['order_count'] = order_count


Loading market history in Jita for 11673 items...
Success!
Processing history content into dataframe...
Done!
Pulling out historical volume data for 30 days...
Pulling out historical price data for 30 days...
Pulling out historical order count data for 30 days...
Done!


In [12]:
#bogus_threshold = 100

# Remove orders where the ratio of current price to historic price is too high
#summary = summary[(summary['Sell price'] / price) < float(bogus_threshold)]

# Finally, calculate expected profit per item by multiplying unit profit by mean daily volume or number of available units, whichever is lower
summary['Total Profit'] = summary['Profit'] * summary.apply(lambda row: min(row['Units available'], row['Daily Volume']), axis=1)

summary['Daily Profit'] = summary['Profit'] * summary['Daily Volume']

# Clean up by dropping NAN rows - Usually this means no volume data or no sell-side orders.
#summary = summary.dropna()

# Sort by profit descending for output
summary = summary.sort_values(by='Daily Profit', ascending=False)

# Use item names instead of typeIDs
summary['typeID'] = summary.index
summary = summary.rename(typeIDs_to_names)
summary.to_csv('exports/summary.csv') 

print (summary)

                                                      Buy price     Sell price         Profit  Margin  buy_competition  sell_competition  Units available  Daily Volume      Avg Price  order_count    Total Profit   Daily Profit  typeID
type_id                                                                                                                                                                                                                                   
23rd Tier Overseer's Personal Effects               35090000.00   120905785.00    85815785.00    2.45              NaN               NaN               68        638.34   133746068.97        89.07   5835473380.00 54780062480.00   19421
Tetrimonic Warclone Blanks                            685000.00      886704.00      201704.00    0.29              NaN               NaN            22928     218528.59      956424.14      1485.52   4624669312.00 44078089952.28   85178
Overseer Sabik Vial                                   652000

In [13]:
test_summary = summary

#test_summary['Trade score'] = test_summary['Profit'] * (test_summary['Daily Volume']/10) / ((test_summary['buy_competition']+test_summary['sell_competition'])*10000)
# TODO look up how to do weighted scoring
test_summary['Trade score'] =(
    test_summary['Buy price']*0.001+
    test_summary['Daily Volume']*0.14+
    test_summary['Margin']*0.85
    /
    (test_summary['buy_competition']+
     test_summary['sell_competition'])
)


In [14]:
test_summary['Trade score'] =(
    test_summary['Buy price']*0.0+
    test_summary['Daily Volume']*0.5+
    test_summary['Margin']*0.5
    #/
    #(test_summary['buy_competition']+
    # test_summary['sell_competition'])
)


In [15]:
test_summary = test_summary.sort_values(by='Trade score', ascending=False)
print(test_summary)

                                               Buy price     Sell price         Profit          Margin  buy_competition  sell_competition  Units available  Daily Volume     Avg Price  order_count    Total Profit  Daily Profit  typeID     Trade score
type_id                                                                                                                                                                                                                                                  
Estamel's Modified 425mm Railgun                    0.01  2770026350.00  2770026349.99 277002634999.00              NaN               NaN             1000          0.21 3124833333.33         6.00    573108900.00  573108900.00   14405 138501317499.60
Tairei's Modified Dual Heavy Pulse Laser            0.01  2215836350.00  2215836349.99 221583634999.00              NaN               NaN             1000          0.00           NaN          NaN            0.00          0.00   14425 110791817499.50


In [16]:

output_summary = test_summary[['Buy price', 'Sell price','Profit', 'Margin',
        'Daily Volume', 'Daily Profit', 
        'Trade score'
]]
print(output_summary)
# Output to a pretty table and return URL for further use
url = render.render_to_html(output_summary)

                                               Buy price     Sell price         Profit          Margin  Daily Volume  Daily Profit     Trade score
type_id                                                                                                                                           
Estamel's Modified 425mm Railgun                    0.01  2770026350.00  2770026349.99 277002634999.00          0.21  573108900.00 138501317499.60
Tairei's Modified Dual Heavy Pulse Laser            0.01  2215836350.00  2215836349.99 221583634999.00          0.00          0.00 110791817499.50
Kaikka's Modified 425mm Railgun                     0.01   821217215.00   821217214.99  82121721499.00          0.00          0.00  41060860749.50
Mizuro's Modified Cruise Missile Launcher           0.01   616998200.00   616998199.99  61699819999.00          0.00          0.00  30849909999.50
CONCORD Dual 1000mm Railgun                         0.01   577188885.00   577188884.99  57718888499.00          1.41  

[15:17:52] PaIadin > <url=showinfo:35833//1044752365771>Perimeter - 0.0% Fence Tax Haven</url>
